# 13 Reproducible Research — A Minimal Verifiable Workflow

Using the Songbai Nursing Home Legionnaires' disease data to demonstrate a reproducible "from zero to summary" workflow.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# --- Step 1: Read the data and produce a summary ---
path = Path("data/synthetic/legionella_outbreak.csv")
df = pd.read_csv(path)
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

summary = {
    "n_residents": len(df),
    "n_zones": df.groupby(["floor", "wing"]).ngroups,
    "n_infected": int(df["infected"].sum()),
    "n_deaths": int((df["outcome"] == "dead").sum()),
    "attack_rate": f"{df['infected'].mean():.1%}",
    "cfr": f"{(df['outcome'] == 'dead').sum() / df['infected'].sum():.1%}",
}

print("=== Outbreak Summary ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\n-> This dict is our minimal verifiable result")
print("-> Anyone on any machine should get the exact same numbers")

In [ ]:
# --- Step 2: Reproducibility checklist ---
from pathlib import Path as _P

checks = {
    "uv.lock exists": _P("uv.lock").exists(),
    "data file exists": _P("data/synthetic/legionella_outbreak.csv").exists(),
    "pyproject.toml exists": _P("pyproject.toml").exists(),
    "tests/ directory exists": _P("tests").is_dir(),
}

print("=== Reproducibility Checklist ===")
for item, ok in checks.items():
    status = "✓" if ok else "✗"
    print(f"  [{status}] {item}")

all_pass = all(checks.values())
print(f"\n-> {'All checks passed! The environment is reproducible' if all_pass else 'Some checks failed and need fixing'}")

In [ ]:
# --- Step 3: Write the summary to CSV ---
import json

# Method 1: save as CSV
summary_df = pd.DataFrame([summary])
output_path = Path("data/processed")
output_path.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_path / "summary.csv", index=False)

# Method 2: save as JSON (preserves types)
with open(output_path / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("=== Output Files ===")
print(f"  CSV: {output_path / 'summary.csv'}")
print(f"  JSON: {output_path / 'summary.json'}")
print("\n-> Next time you verify, compare these files to see if the results match")

In [ ]:
# --- Step 4: Record version information ---
import sys
import platform

env_info = {
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
}

print("=== Environment Version Info ===")
for k, v in env_info.items():
    print(f"  {k}: {v}")

print("\n-> Attach the version info to your report so others can reproduce your environment")
print("-> uv.lock can automatically pin every package version for you")

## Summary

| Step | Skill Learned |
|------|------------|
| Read + summarize | Produce a minimal verifiable result with a dict |
| Checklist | Confirm all environment files are present |
| Save output | Preserve results as CSV / JSON for comparison |
| Version info | Record Python / package versions |

**The three pillars of reproducibility**:
1. **Data**: a fixed input file (`legionella_outbreak.csv`)
2. **Code**: version control (git commit)
3. **Environment**: pinned packages (`uv.lock`)

In the next chapter (Ch14), we'll integrate all these skills into one complete real-world case study.